In [1]:
import infomeasure as im
import numpy as np
import matplotlib.pyplot as plt
import enum
import os
import pandas as pd
from tqdm import tqdm



class BrainMasterMontage(enum.Enum):
    Fp1 = 1
    F3 = 2
    C3 = 3
    P3 = 4
    O1 = 5
    F7 = 6
    T3 = 7
    T5 = 8
    Fz = 9
    Fp2 = 10
    F4 = 11
    C4 = 12
    P4 = 13
    O2 = 14
    F8 = 15
    T4 = 16
    T6 = 17
    Cz = 18
    Pz = 19

In [2]:
def get_channels_pairs_dict():
    """
    Returns a dict of all ordered pairs of BrainMasterMontage channels.
    Key: 'ChX->ChY', Value: [idx_of_x, idx_of_x] (0-based indices)
    """
    electrodeNames = [e.name for e in BrainMasterMontage]
    pairsDict = {}
    for idxY, chY in enumerate(electrodeNames):
        for idxX, chX in enumerate(electrodeNames):
            if idxX != idxY:
                key = f"{chX}->{chY}"
                pairsDict[key] = [idxX, idxY]
    return pairsDict

In [3]:
pairsDict = get_channels_pairs_dict()
print(f"Pairs dict: {pairsDict}")
print(f"Number of pairs: {len(pairsDict)}")
print(list(pairsDict.keys())[:10])

Pairs dict: {'F3->Fp1': [1, 0], 'C3->Fp1': [2, 0], 'P3->Fp1': [3, 0], 'O1->Fp1': [4, 0], 'F7->Fp1': [5, 0], 'T3->Fp1': [6, 0], 'T5->Fp1': [7, 0], 'Fz->Fp1': [8, 0], 'Fp2->Fp1': [9, 0], 'F4->Fp1': [10, 0], 'C4->Fp1': [11, 0], 'P4->Fp1': [12, 0], 'O2->Fp1': [13, 0], 'F8->Fp1': [14, 0], 'T4->Fp1': [15, 0], 'T6->Fp1': [16, 0], 'Cz->Fp1': [17, 0], 'Pz->Fp1': [18, 0], 'Fp1->F3': [0, 1], 'C3->F3': [2, 1], 'P3->F3': [3, 1], 'O1->F3': [4, 1], 'F7->F3': [5, 1], 'T3->F3': [6, 1], 'T5->F3': [7, 1], 'Fz->F3': [8, 1], 'Fp2->F3': [9, 1], 'F4->F3': [10, 1], 'C4->F3': [11, 1], 'P4->F3': [12, 1], 'O2->F3': [13, 1], 'F8->F3': [14, 1], 'T4->F3': [15, 1], 'T6->F3': [16, 1], 'Cz->F3': [17, 1], 'Pz->F3': [18, 1], 'Fp1->C3': [0, 2], 'F3->C3': [1, 2], 'P3->C3': [3, 2], 'O1->C3': [4, 2], 'F7->C3': [5, 2], 'T3->C3': [6, 2], 'T5->C3': [7, 2], 'Fz->C3': [8, 2], 'Fp2->C3': [9, 2], 'F4->C3': [10, 2], 'C4->C3': [11, 2], 'P4->C3': [12, 2], 'O2->C3': [13, 2], 'F8->C3': [14, 2], 'T4->C3': [15, 2], 'T6->C3': [16, 2], 'Cz

In [97]:
dataDir = "/Users/wachiii/Workschii/brain-asd/data/data_children_no_artifact/trimedData/age5-8/asd"
pairsDict = get_channels_pairs_dict()


resultDict = {
    "Pair name": [],
    "TE list (without gaussian noise)": [],
    "Avg TE (without gaussian noise)": [],
    "TE list (with gaussian noise)": [],
    "Avg TE (with gaussian noise)": [],
}


for pairName, (idxX, idxY) in tqdm(pairsDict.items(), desc="Pairs"):
    teList = []
    teListNoise = []
    for fname in tqdm(os.listdir(dataDir), desc=f"{pairName}", leave=False):
        if fname.endswith(".npy"):
            fpath = os.path.join(dataDir, fname)
            data = np.load(fpath)
            chX = data[idxX, :]
            chY = data[idxY, :]
            try:
                te = im.transfer_entropy(chX, chY, approach="metric")
            except Exception:
                te = np.nan
            try:
                teNoise = im.transfer_entropy(chX, chY, approach="metric", noise_level=0.001)
            except Exception:
                teNoise = np.nan
            teList.append(te)
            teListNoise.append(teNoise)
    resultDict["Pair name"].append(pairName)
    resultDict["TE list (without gaussian noise)"].append(teList)
    avg = np.nanmean(teList)
    std = np.nanstd(teList)
    resultDict["Avg TE (without gaussian noise)"].append(f"{avg:.4f} ± {std:.4f}")
    resultDict["TE list (with gaussian noise)"].append(teListNoise)
    avgNoise = np.nanmean(teListNoise)
    stdNoise = np.nanstd(teListNoise)
    resultDict["Avg TE (with gaussian noise)"].append(f"{avgNoise:.4f} ± {stdNoise:.4f}")


df = pd.DataFrame(resultDict)
df.to_excel("transfer_entropy_results_58_asd.xlsx", index=False)
print("Saved results to transfer_entropy_results.xlsx")

Pairs: 100%|██████████| 342/342 [9:36:14<00:00, 101.09s/it]  

Saved results to transfer_entropy_results.xlsx


In [4]:
dataDir = "/Users/wachiii/Workschii/brain-asd/data/data_children_no_artifact/trimedData/age5-8/hc"
pairsDict = get_channels_pairs_dict()


resultDict = {
    "Pair name": [],
    "TE list (without gaussian noise)": [],
    "Avg TE (without gaussian noise)": [],
    "TE list (with gaussian noise)": [],
    "Avg TE (with gaussian noise)": [],
}


for pairName, (idxX, idxY) in tqdm(pairsDict.items(), desc="Pairs"):
    teList = []
    teListNoise = []
    for fname in tqdm(os.listdir(dataDir), desc=f"{pairName}", leave=False):
        if fname.endswith(".npy"):
            fpath = os.path.join(dataDir, fname)
            data = np.load(fpath)
            chX = data[idxX, :]
            chY = data[idxY, :]
            try:
                te = im.transfer_entropy(chX, chY, approach="metric")
            except Exception:
                te = np.nan
            try:
                teNoise = im.transfer_entropy(chX, chY, approach="metric", noise_level=0.001)
            except Exception:
                teNoise = np.nan
            teList.append(te)
            teListNoise.append(teNoise)
    resultDict["Pair name"].append(pairName)
    resultDict["TE list (without gaussian noise)"].append(teList)
    avg = np.nanmean(teList)
    std = np.nanstd(teList)
    resultDict["Avg TE (without gaussian noise)"].append(f"{avg:.4f} ± {std:.4f}")
    resultDict["TE list (with gaussian noise)"].append(teListNoise)
    avgNoise = np.nanmean(teListNoise)
    stdNoise = np.nanstd(teListNoise)
    resultDict["Avg TE (with gaussian noise)"].append(f"{avgNoise:.4f} ± {stdNoise:.4f}")


df = pd.DataFrame(resultDict)
df.to_excel("transfer_entropy_results_58_hc.xlsx", index=False)
print("Saved results to transfer_entropy_results.xlsx")

Pairs: 100%|██████████| 342/342 [7:07:24<00:00, 74.98s/it]  

Saved results to transfer_entropy_results.xlsx


In [99]:
dataDir = "/Users/wachiii/Workschii/brain-asd/data/data_children_no_artifact/trimedData/age9-12/asd"
pairsDict = get_channels_pairs_dict()


resultDict = {
    "Pair name": [],
    "TE list (without gaussian noise)": [],
    "Avg TE (without gaussian noise)": [],
    "TE list (with gaussian noise)": [],
    "Avg TE (with gaussian noise)": [],
}


for pairName, (idxX, idxY) in tqdm(pairsDict.items(), desc="Pairs"):
    teList = []
    teListNoise = []
    for fname in tqdm(os.listdir(dataDir), desc=f"{pairName}", leave=False):
        if fname.endswith(".npy"):
            fpath = os.path.join(dataDir, fname)
            data = np.load(fpath)
            chX = data[idxX, :]
            chY = data[idxY, :]
            try:
                te = im.transfer_entropy(chX, chY, approach="metric")
            except Exception:
                te = np.nan
            try:
                teNoise = im.transfer_entropy(chX, chY, approach="metric", noise_level=0.001)
            except Exception:
                teNoise = np.nan
            teList.append(te)
            teListNoise.append(teNoise)
    resultDict["Pair name"].append(pairName)
    resultDict["TE list (without gaussian noise)"].append(teList)
    avg = np.nanmean(teList)
    std = np.nanstd(teList)
    resultDict["Avg TE (without gaussian noise)"].append(f"{avg:.4f} ± {std:.4f}")
    resultDict["TE list (with gaussian noise)"].append(teListNoise)
    avgNoise = np.nanmean(teListNoise)
    stdNoise = np.nanstd(teListNoise)
    resultDict["Avg TE (with gaussian noise)"].append(f"{avgNoise:.4f} ± {stdNoise:.4f}")


df = pd.DataFrame(resultDict)
df.to_excel("transfer_entropy_results_912_asd.xlsx", index=False)
print("Saved results to transfer_entropy_results.xlsx")

Pairs: 100%|██████████| 342/342 [6:33:38<00:00, 69.06s/it]   

Saved results to transfer_entropy_results.xlsx


In [100]:
dataDir = "/Users/wachiii/Workschii/brain-asd/data/data_children_no_artifact/trimedData/age9-12/hc"
pairsDict = get_channels_pairs_dict()


resultDict = {
    "Pair name": [],
    "TE list (without gaussian noise)": [],
    "Avg TE (without gaussian noise)": [],
    "TE list (with gaussian noise)": [],
    "Avg TE (with gaussian noise)": [],
}


for pairName, (idxX, idxY) in tqdm(pairsDict.items(), desc="Pairs"):
    teList = []
    teListNoise = []
    for fname in tqdm(os.listdir(dataDir), desc=f"{pairName}", leave=False):
        if fname.endswith(".npy"):
            fpath = os.path.join(dataDir, fname)
            data = np.load(fpath)
            chX = data[idxX, :]
            chY = data[idxY, :]
            try:
                te = im.transfer_entropy(chX, chY, approach="metric")
            except Exception:
                te = np.nan
            try:
                teNoise = im.transfer_entropy(chX, chY, approach="metric", noise_level=0.001)
            except Exception:
                teNoise = np.nan
            teList.append(te)
            teListNoise.append(teNoise)
    resultDict["Pair name"].append(pairName)
    resultDict["TE list (without gaussian noise)"].append(teList)
    avg = np.nanmean(teList)
    std = np.nanstd(teList)
    resultDict["Avg TE (without gaussian noise)"].append(f"{avg:.4f} ± {std:.4f}")
    resultDict["TE list (with gaussian noise)"].append(teListNoise)
    avgNoise = np.nanmean(teListNoise)
    stdNoise = np.nanstd(teListNoise)
    resultDict["Avg TE (with gaussian noise)"].append(f"{avgNoise:.4f} ± {stdNoise:.4f}")


df = pd.DataFrame(resultDict)
df.to_excel("transfer_entropy_results_912_hc.xlsx", index=False)
print("Saved results to transfer_entropy_results.xlsx")

Pairs:   3%|▎         | 11/342 [09:18<4:40:18, 50.81s/it]


KeyboardInterrupt: 

In [ ]:
print(resultDict)

{'Pair name': ['F3->Fp1', 'C3->Fp1', 'P3->Fp1'], 'TE list (without gaussian noise)': [[np.float64(0.5062781573633757), np.float64(0.5165468838206554), np.float64(0.5016987429493345), np.float64(0.530529037198652), np.float64(0.5128236835306708), np.float64(0.5045660175444201), np.float64(0.508914193035889), np.float64(0.5215042914007414), np.float64(0.5185872486235181), np.float64(0.5106249431693698), np.float64(0.4997933224591376), np.float64(0.5018564970208159), np.float64(0.5317899159807357), np.float64(0.5089708921566072), np.float64(0.5071258359656735), np.float64(0.5140052631122496), np.float64(0.5456712219136567), np.float64(0.517336782207416), np.float64(0.5272237285664275), np.float64(0.5128881984085449)], [np.float64(0.5007542951985509), np.float64(0.5111329561891299), np.float64(0.5022876185050373), np.float64(0.5242517888369383), np.float64(0.5112425778383682), np.float64(0.4940571977613448), np.float64(0.5059324124924337), np.float64(0.5216449994172646), np.float64(0.50755

In [ ]:
df = pd.DataFrame(resultDict)
df.to_csv("transfer_entropy_results.csv", index=False)
df.to_excel("transfer_entropy_results.xlsx", index=False)